In [1]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

In [2]:
data_dir = '../../../data/processed/Mux-VisiumHD-HE'

ad_mux_rna = sc.read_h5ad(join(data_dir, 'Mux/ad_rna.h5ad'))
ad_mux_atac = sc.read_h5ad(join(data_dir, 'Mux/ad_atac.h5ad'))

ad_vhd_rna = sc.read_h5ad(join(data_dir, 'VisiumHD/ad_rna.h5ad'))
ad_vhe_rna = sc.read_h5ad(join(data_dir, 'VisiumHE/ad_rna.h5ad')) 

input_dict = { 
    'rna':  [ad_mux_rna,  ad_vhd_rna,  ad_vhe_rna],
    'atac':  [ad_mux_atac, None       , None],
}

input_key = 'dimred_bc'
batch_key = 'Slice'

In [3]:
cache_dir = './cache_dir/Mux-VisiumHD-HE'
df_cca = pd.read_csv(join(cache_dir, 'cca_coordinates.csv'), index_col=0)    # exported using seurat integration pipeline 
for adx in input_dict['rna']:
    adx.obsm[input_key] = df_cca.loc[adx.obs_names].values

Epigenome_preprocess(input_dict['atac'], batch_corr=False, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)   

In [4]:
def stack(xl, key):
    xs, ns = [], []
    for adx in xl:
        if adx is not None:
            xs.append(adx.obsm[key])
            ns.append(adx.obs_names)
    df = pd.DataFrame(np.vstack(xs), index=np.hstack(ns))
    return df

for m1, m2 in zip(['rna', 'atac'], ['RNA', 'ATAC']):
    fig_dir = f'../../../results/embeddings/Leiden-{m2}/Mux-VisiumHD-HE'
    os.makedirs(fig_dir, exist_ok=True)
    df = stack(input_dict[m1], input_key)
    df.to_csv(join(fig_dir, 'df_emb.csv'))